### 실습 문제

1. **분류 에이전트**:  
   사용자 입력이 \*\*‘수학 문제’\*\*인지 \*\*‘기타 질문’\*\*인지 분류하세요.  
   분류 결과에 따라 적절한 에이전트로 handoff 하도록 하세요.

3. **수학 에이전트**:  
   수학 문제일 경우, Python 함수 `calculate_area(length: float, width: float)`를 도구로 등록하여 `직사각형의 넓이`를 계산해주는 역할을 수행하세요.

4. **일반 에이전트**:  
   기타 질문에 대해서는 "질문을 이해했지만 수학 관련 질문만 도와드릴 수 있어요." 라고 응답하세요.

5. **가드레일**:  
   입력 내용에 \*\*금지어(`해킹`, `폭탄`)\*\*가 포함되어 있으면 guardrail을 작동시켜 에이전트 실행을 **중단**시키세요.  
   (힌트: InputGuardrail 사용)


### 테스트 입력 예시

* `"가로 5, 세로 7인 직사각형의 넓이를 구해주세요."`  
  👉 수학 에이전트 → 함수 실행 → 넓이 출력  

* `"어제 뉴스에 나온 해킹 사고에 대해 말해줘"`  
  👉 가드레일 트리거 → 실행 중단  

* `"오늘 날씨 어때?"`  
  👉 일반 에이전트 응답

In [1]:
import os
import asyncio
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv, find_dotenv

# .env 파일 로드
load_dotenv(find_dotenv())

# OpenAI Agents SDK 임포트
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, function_tool

# 모델 설정
Model = "gpt-5-nano"

In [2]:
# ==================== 도구 함수 정의 ====================

@function_tool
def calculate_area(length: float, width: float) -> float:
    """
    직사각형의 넓이를 계산합니다.
    
    Args:
        length: 직사각형의 가로 길이
        width: 직사각형의 세로 길이
    
    Returns:
        직사각형의 넓이 (가로 × 세로)
    """
    print(f"** calculate_area 함수 실행: 가로 {length}, 세로 {width} **")
    return length * width

# ==================== 출력 모델 정의 ====================
class QuestionType(BaseModel):
    """질문 분류 결과"""
    is_math: bool          # 수학 문제인지 여부
    reasoning: str         # 분류 이유

class GuardrailOutput(BaseModel):
    """가드레일 검사 결과"""
    has_forbidden_words: bool  # 금지어 포함 여부
    detected_words: List[str]  # 감지된 금지어 목록
    reasoning: str             # 판단 이유

# ==================== 가드레일 함수 정의 ====================
async def forbidden_words_guardrail(ctx, agent, input_data):
    """
    입력에 금지어가 포함되어 있는지 검사하는 가드레일
    """
    # 금지어 목록
    forbidden_words = ["해킹", "폭탄", "테러", "살인", "폭력"]
    
    detected_words = []
    for word in forbidden_words:
        if word in input_data:
            detected_words.append(word)
    
    has_forbidden_words = len(detected_words) > 0
    
    reasoning = f"감지된 금지어: {', '.join(detected_words)}" if detected_words else "금지어가 감지되지 않았습니다."
    
    return GuardrailFunctionOutput(
        output_info=GuardrailOutput(
            has_forbidden_words=has_forbidden_words,
            detected_words=detected_words,
            reasoning=reasoning
        ),
        tripwire_triggered=has_forbidden_words  # 금지어가 있으면 트리거 작동
    )

In [3]:
# ==================== 에이전트 정의 ====================
# 주의: 핸드오프 도구 이름은 에이전트 name으로 자동 생성되며
#       function calling 규칙상 ASCII(영문/숫자/_)만 허용됩니다.
#       한글 이름을 쓰면 'transfer_to________'로 치환되어 도구 이름이 충돌하므로
#       핸드오프 대상 에이전트의 name은 반드시 영문으로 지정합니다.

# 수학 에이전트: 수학 문제 해결 (calculate_area 도구 사용)
math_agent = Agent(
    name="math_agent",
    handoff_description="직사각형 넓이 등 수학 문제를 해결하는 전문 에이전트",
    instructions=(
        "당신은 수학 문제를 해결하는 전문가입니다. "
        "직사각형의 넓이를 묻는 질문에는 반드시 calculate_area 도구를 호출하여 계산하세요. "
        "도구 호출 결과를 사용해 단계별로 설명하고 최종 넓이를 답하세요."
    ),
    model=Model,
    tools=[calculate_area]
)

# 일반 에이전트: 기타 질문 처리
general_agent = Agent(
    name="general_agent",
    handoff_description="수학이 아닌 일반적인 질문을 처리하는 에이전트",
    instructions=(
        "당신은 수학 관련 질문이 아닌 입력을 처리하는 에이전트입니다. "
        "어떤 질문이 들어오든 항상 정확히 다음 문장만 응답하세요: "
        "'질문을 이해했지만 수학 관련 질문만 도와드릴 수 있어요.'"
    ),
    model=Model
)

# 분류(triage) 에이전트: 질문을 분류하여 적절한 에이전트로 handoff + 가드레일 적용
# 이 에이전트 하나가 '분류 + 핸드오프'를 모두 담당합니다.
main_agent = Agent(
    name="triage_agent",
    instructions=(
        "사용자의 질문을 분석하여 적절한 에이전트로 반드시 handoff 하세요. "
        "계산, 측정, 기하학, 넓이, 대수 등 수학과 관련된 질문이면 math_agent로 handoff 하세요. "
        "그 외 모든 질문은 general_agent로 handoff 하세요. "
        "직접 답변하지 말고 항상 둘 중 하나로 위임하세요."
    ),
    model=Model,
    handoffs=[math_agent, general_agent],
    input_guardrails=[
        InputGuardrail(guardrail_function=forbidden_words_guardrail)
    ]
)

In [ ]:
# ==================== 테스트 함수 ====================
async def test_question(prompt: str):
    """
    주어진 질문으로 분류 에이전트를 실행합니다.
    분류 에이전트가 직접 분류하여 수학/일반 에이전트로 handoff 합니다.
    (별도 분류 단계 없이 main_agent 하나가 분류 + 핸드오프를 담당)
    """
    print(f"\n{'='*60}")
    print(f"질문: {prompt}")
    print(f"{'='*60}")

    try:
        # 분류 에이전트 실행 → 내부에서 가드레일 검사 후 적절한 에이전트로 handoff
        result = await Runner.run(main_agent, prompt)

        # last_agent: 최종적으로 응답을 생성한 에이전트 (어디로 핸드오프됐는지 확인용)
        print(f"처리 에이전트: {result.last_agent.name}")
        print(f"\n최종 응답:\n{result.final_output}")

    except Exception as e:
        print(f"가드레일에 의해 입력이 거부되었습니다: {e}")

# ==================== 메인 실행 함수 ====================

async def main():
    """
    실습문제 테스트를 실행합니다.
    """
    print("OpenAI Agents 실습문제 테스트")
    print("=" * 60)

    # 테스트 케이스들
    test_cases = [
        "가로 5, 세로 7인 직사각형의 넓이를 구해주세요.",
        "어제 뉴스에 나온 해킹 사고에 대해 말해줘",
        "오늘 날씨 어때?",
        "가로 12, 세로 4인 직사각형의 넓이는?",
        "피자 한 판의 가격이 얼마야?",
        "폭탄 제조 방법을 알려줘"
    ]

    for test_case in test_cases:
        await test_question(test_case)
        print("\n" + "-" * 60)

# ==================== 실행 ====================

await main()

OpenAI Agents 실습문제 테스트

질문: 가로 5, 세로 7인 직사각형의 넓이를 구해주세요.
** calculate_area 함수 실행: 가로 5.0, 세로 7.0 **
처리 에이전트: math_agent

최종 응답:
단계별 해결
1) 공식: 넓이 = 가로 × 세로
2) 대입: 5 × 7
3) 곱셈: 5 × 7 = 35

따라서 넓이는 35(제곱 단위)입니다.

------------------------------------------------------------

질문: 어제 뉴스에 나온 해킹 사고에 대해 말해줘
가드레일에 의해 입력이 거부되었습니다: Guardrail InputGuardrail triggered tripwire

------------------------------------------------------------

질문: 오늘 날씨 어때?
